In [1]:
!pip install -q langchain langgraph pypdf pandas langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
!pip install -q langdetect
# Installation de l'outil de détection de langue
!pip install -q python-docx


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.9 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd
import json
import xml.etree.ElementTree as ET
from pypdf import PdfReader
from docx import Document
from langdetect import detect_langs

class TextIngestionAgent:
    def __init__(self):
        self.reset()
        self.supported_formats = [".txt", ".pdf", ".csv", ".json", ".xml", ".docx", ".log", ".py", ".js"]

    def reset(self):
        self.raw_texts = []
        self.unique_texts = set()
        self.languages = set()
        self.formats = set()

    def normalize_encoding(self, text):
        return text.encode('utf-8', 'ignore').decode('utf-8')

    def scan_file(self, file_path):
        ext = os.path.splitext(file_path)[1].lower()
        self.formats.add(ext.replace('.', ''))
        text = ""

        try:
            if ext == ".txt" or ext == ".log" or ext in [".py", ".js"]: # txt, logs, code
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read()

            elif ext == ".pdf":
                reader = PdfReader(file_path)
                text = " ".join([page.extract_text() or "" for page in reader.pages])

            elif ext == ".docx":
                doc = Document(file_path)
                text = " ".join([para.text for para in doc.paragraphs])

            elif ext == ".json":
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    text = json.dumps(data) # On convertit tout le JSON en texte pour le NLP

            elif ext == ".csv":
                df = pd.read_csv(file_path)
                text = df.to_string()

            elif ext == ".xml":
                tree = ET.parse(file_path)
                root = tree.getroot()
                text = "".join(root.itertext())

            # Traitement
            normalized_text = self.normalize_encoding(text).strip()
            self.raw_texts.append(normalized_text)
            self.unique_texts.add(normalized_text)

            # Langue
            predictions = detect_langs(normalized_text[:1000]) # Analyse sur le début
            for lang in predictions:
                if lang.prob > 0.5: self.languages.add(lang.lang)

            return normalized_text

        except Exception as e:
            return f"Erreur sur {file_path}: {str(e)}"

    def get_report(self):
        total_docs = len(self.raw_texts)
        dups = total_docs - len(self.unique_texts)
        all_words = sum(len(t.split()) for t in self.unique_texts)
        avg = all_words / len(self.unique_texts) if self.unique_texts else 0
        return {
            "total_docs": total_docs,
            "languages": sorted(list(self.languages)),
            "formats": sorted(list(self.formats)),
            "duplicates": dups,
            "avg_length": f"{int(avg)} words"
        }

ingestion_agent = TextIngestionAgent()

In [4]:
# 1. On scanne le même fichier deux fois pour tester les doublons

# ingestion_agent.reset() # On vide tout avant
ingestion_agent.scan_file("test.txt")
ingestion_agent.scan_file("test.txt")

# 2. Affichage du rapport final
import json
report = ingestion_agent.get_report()

print("--- OUTPUT AGENT 1 (FORMAT JSON) ---")
print(json.dumps(report, indent=4)) # Pour un affichage propre comme dans le doc

--- OUTPUT AGENT 1 (FORMAT JSON) ---
{
    "total_docs": 0,
    "languages": [],
    "formats": [
        "txt"
    ],
    "duplicates": 0,
    "avg_length": "0 words"
}


In [5]:
!pip install -q pypdf

In [6]:
# 1. On lance le scan sur le fichier PDF
# Assure-toi que le nom du fichier correspond bien à ce qui est dans ton dossier à gauche

# ingestion_agent.reset() # On vide tout avant
contenu_pdf = ingestion_agent.scan_file("nlp dataset intelligence engine.pdf")

# 2. On affiche le rapport
import json
report = ingestion_agent.get_report()

print("--- RÉSULTAT DU TEST PDF ---")
print(f"Extrait du contenu : {contenu_pdf[:200]}...") # On affiche les 200 premiers caractères
print("\n--- RAPPORT AGENT 1 ---")
print(json.dumps(report, indent=4))

--- RÉSULTAT DU TEST PDF ---
Extrait du contenu : Erreur sur nlp dataset intelligence engine.pdf: [Errno 2] No such file or directory: 'nlp dataset intelligence engine.pdf'...

--- RAPPORT AGENT 1 ---
{
    "total_docs": 0,
    "languages": [],
    "formats": [
        "pdf",
        "txt"
    ],
    "duplicates": 0,
    "avg_length": "0 words"
}


In [7]:
code_content = """
# Ce script calcule la somme de deux nombres
# This is a simple python function
def add(a, b):
    result = a + b
    print("Calcul terminé")
    return result

# Calling the function
add(5, 10)
"""

with open("script.py", "w", encoding="utf-8") as f:
    f.write(code_content)

print("Fichier script.py créé avec succès.")

✅ Fichier script.py créé avec succès.


In [8]:
# On vide la mémoire pour un test précis
ingestion_agent.reset()

# Scan du fichier source
ingestion_agent.scan_file("script.py")

# Affichage du rapport
import json
print("--- RAPPORT AGENT 1 (CODE SOURCE) ---")
print(json.dumps(ingestion_agent.get_report(), indent=4))

--- RAPPORT AGENT 1 (CODE SOURCE) ---
{
    "total_docs": 1,
    "languages": [
        "fr"
    ],
    "formats": [
        "py"
    ],
    "duplicates": 0,
    "avg_length": "34 words"
}


In [9]:
!pip install -q -U sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 60.9 MB/s eta 0:00:00


In [10]:
import numpy as np
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from collections import Counter
import re

class DatasetProfilerNLP:
    def __init__(self):
        # Chargement du modèle pour les Embeddings (BER)
        # 'all-MiniLM-L6-v2' est rapide et excellent pour le clustering
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def calculate_noise(self, text):
        """Détecte le bruit : ratio de caractères spéciaux / texte"""
        special_chars = len(re.findall(r'[^a-zA-Z0-9\sàâäéèêëîïôöùûüç]', text))
        return (special_chars / len(text)) if len(text) > 0 else 0

    def get_structure(self, text):
        """Détermine la structure : dialogue, code ou paragraphe"""
        if any(kw in text for kw in ["def ", "import ", "function", "{", "}", "-->"]):
            return "Code source / Technique"
        elif text.count(':') > len(text.split('.')) and len(text) < 500:
            return "Dialogue / Chat conversation"
        else:
            return "Paragraphes structurés"

    def analyze_dataset(self, texts):
        if not texts: return "Aucune donnée à analyser."

        # 1. Analyse statistique (Longueur et Vocabulaire)
        all_words = " ".join(texts).lower().split()
        avg_len = sum(len(t.split()) for t in texts) / len(texts)
        vocab_richness = len(set(all_words)) / len(all_words) if all_words else 0

        # 2. Embeddings et Clustering (Groupes thématiques)
        # On transforme les textes en vecteurs numériques
        embeddings = self.embedder.encode(texts)
        # On crée 2 groupes (clusters) par défaut pour voir les thèmes
        num_clusters = min(len(texts), 3)
        kmeans = KMeans(n_clusters=num_clusters, n_init=10).fit(embeddings)

        # 3. Détection de la Nature du Dataset
        noise_level = sum(self.calculate_noise(t) for t in texts) / len(texts)
        structure = self.get_structure(texts[0]) # Analyse du premier doc comme échantillon

        # Logique de décision pour le "Dataset probable"
        prob_type = "Général"
        if "Code" in structure: prob_type = "Code Source / Documentation"
        elif "Dialogue" in structure: prob_type = "Conversations Chat / Support"
        elif any(w in all_words for w in ["patient", "traitement", "diagnostic"]): prob_type = "Medical"

        if noise_level > 0.3: prob_type += " (Haute présence de Bruit/Spam)"

        # --- OUTPUT CONFORME AU CAHIER DES CHARGES ---
        return {
            "Analyse": {
                "avg_length": f"{int(avg_len)} mots",
                "vocab_richness": f"{int(vocab_richness*100)}%",
                "noise_level": "Faible" if noise_level < 0.1 else "Élevé",
                "structure": structure
            },
            "Embeddings_Clustering": {
                "method": "Sentence-BERT + KMeans",
                "thematic_groups": num_clusters,
                "anomalies_detected": "Oui" if noise_level > 0.4 else "Non"
            },
            "Dataset_probable": prob_type
        }

# Initialisation
profiler_nlp = DatasetProfilerNLP()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
# 1. L'Agent 1 récupère le texte d'un fichier
texte_brut = ingestion_agent.scan_file("script.py")

# 2. L'Agent 2 analyse TOUS les textes récupérés par l'Agent 1
rapport_final = profiler_nlp.analyze_dataset(ingestion_agent.raw_texts)

import json
print(json.dumps(rapport_final, indent=4, ensure_ascii=False))

{
    "Analyse": {
        "avg_length": "34 mots",
        "vocab_richness": "42%",
        "noise_level": "Faible",
        "structure": "Code source / Technique"
    },
    "Embeddings_Clustering": {
        "method": "Sentence-BERT + KMeans",
        "thematic_groups": 2,
        "anomalies_detected": "Non"
    },
    "Dataset_probable": "Code Source / Documentation"
}


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1336: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [12]:
!pip install -q fpdf # Installation d'un créateur de PDF léger seullemnt un test

from fpdf import FPDF

# Création du contenu médical
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, txt="RAPPORT MÉDICAL DE SYNTHÈSE", ln=True, align='C')
pdf.ln(10)
pdf.multi_cell(0, 10, txt="""Patient: Jean Dupont.
Diagnostic: Présence de symptômes grippaux avec fièvre élevée (39°C).
Traitement préconisé: Repos strict et prise de paracétamol toutes les 6 heures.
Le patient doit reconsulter en cas de difficultés respiratoires.""")

pdf.output("rapport_medical.pdf")
print(" Fichier 'rapport_medical.pdf' généré avec succès !")

  Preparing metadata (setup.py) ... done
✅ Fichier 'rapport_medical.pdf' généré avec succès !


In [13]:
# 1. Nettoyage de la mémoire
ingestion_agent.reset()

# 2. Agent 1 : Scan du PDF
print("Agent 1 : Lecture du PDF...")
texte_extrait = ingestion_agent.scan_file("rapport_medical.pdf")

# 3. Agent 2 : Profiling du texte extrait
print("Agent 2 : Analyse du contenu...")
rapport_final = profiler_nlp.analyze_dataset(ingestion_agent.raw_texts)

# 4. Affichage du résultat
import json
print("\n--- RÉSULTAT DU TEST MÉDICAL ---")
print(json.dumps(rapport_final, indent=4, ensure_ascii=False))

Agent 1 : Lecture du PDF...
Agent 2 : Analyse du contenu...

--- RÉSULTAT DU TEST MÉDICAL ---
{
    "Analyse": {
        "avg_length": "37 mots",
        "vocab_richness": "91%",
        "noise_level": "Faible",
        "structure": "Paragraphes structurés"
    },
    "Embeddings_Clustering": {
        "method": "Sentence-BERT + KMeans",
        "thematic_groups": 1,
        "anomalies_detected": "Non"
    },
    "Dataset_probable": "Medical"
}


# New section